# Notebook 3 — Hypothesis Testing with Permutation Tests

**Hypothesis testing**: a method to ask "could this pattern have
arisen by chance?"

We'll use **permutation tests** — a non-parametric method that
evaluates significance by shuffling data labels to build a null
distribution, rather than assuming any mathematical distribution.

---
## 1. Setup

In [ ]:
import os, math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
from scipy.stats import hypergeom

sns.set_theme()
np.random.seed(42)

DATA = os.path.join('..', 'data')
SUB_DIR = os.path.join(DATA, 'subfamilies')

def read_fasta(path):
    seqs = {}
    header = None
    with open(path) as f:
        for line in f:
            line = line.strip()
            if line.startswith('>'):
                header = line[1:].split()[0]
                seqs[header] = ''
            elif header:
                seqs[header] += line
    return seqs

---
## 2. Enrichment: Colored Balls

Imagine a bag with **100 balls** of 5 colors:

| Color | Count |
|-------|-------|
| Purple | 20 |
| Red | 30 |
| Blue | 25 |
| Green | 15 |
| Yellow | 10 |

You grab a **subset of 15 balls** and count: **8 are purple**.

Purple is 20% of the bag, so you'd expect about 3 purple in 15
draws. But you got 8.

**Is purple enriched in your subset, or was it just accidental?**

### The concept

The large circle is the **population**. The small circle is our **subset**. Notice how purple is over-represented in the subset.

![Population of balls](../figures/enrichment_schema.png "Coloured balls")

### Expected vs Observed

In [ ]:
# The population
population = (['purple'] * 20 + ['red'] * 30 + ['blue'] * 25
              + ['green'] * 15 + ['yellow'] * 10)

# What we observed in our subset of 15
observed = {'purple': 8, 'red': 3, 'blue': 2, 'green': 1, 'yellow': 1}
subset_size = 15

# How many of each color in the full bag
bag = {'purple': 20, 'red': 30, 'blue': 25, 'green': 15, 'yellow': 10}

# Compare expected vs observed
print("Color", "\t", "Expected", "\t", "Observed", "\t", "Fold enrichment")
print("-" * 56)
for color in bag:
    expected = subset_size * bag[color] / 100
    fold = round(observed[color] / expected, 2)
    print(color, "\t", round(expected, 1), "\t\t", observed[color], "\t\t", fold)

### Is it significant? We'll use a permutation test

Randomly grab 15 balls from the bag thousands of times.
How often do we get 8 or more purple?

In [ ]:
n_permutations = 10000
purple_counts = []

for i in range(n_permutations):
    random_grab = np.random.choice(population, size=subset_size,
                                   replace=False)
    purple_counts.append(np.sum(random_grab == 'purple'))

purple_counts = np.array(purple_counts)

# Plot
sns.histplot(purple_counts, bins=range(0, 13), discrete=True,
             color='steelblue')
plt.axvline(8, color='red', linewidth=2, label='Observed: 8 purple')
plt.xlabel('Purple balls in random subset of 15')
plt.legend()
plt.show()

p_perm = np.mean(purple_counts >= 8)
print("Permutation p-value:", p_perm)

---
## 3. Is echolocator body mass different?

Now a biological example. You measure the body mass of 12 mammals.
Five are echolocators, seven are not. The echolocators happen to
have a **higher average mass**. Is this significant, or could it
be explained by chance?

In [ ]:
# Body mass data (kg)
masses = np.array([39, 42, 37, 50, 38, 45, 22, 31, 27, 33, 40, 36])
labels = np.array(['echo'] * 5 + ['non'] * 7)

echo_mean = masses[labels == 'echo'].mean()
non_mean  = masses[labels == 'non'].mean()
observed_diff = echo_mean - non_mean

print(f"Echolocator mean:     {echo_mean:.1f} kg")
print(f"Non-echolocator mean: {non_mean:.1f} kg")
print(f"Observed difference:  {observed_diff:.1f} kg")

In [ ]:
df_mass = pd.DataFrame({'mass': masses, 'group': labels})

sns.boxplot(data=df_mass, x='group', y='mass')
sns.stripplot(data=df_mass, x='group', y='mass', color='black', alpha=0.5)
plt.ylabel('Body mass (kg)')
plt.title(f'Observed difference: {observed_diff:.1f} kg')
plt.show()

### Exercise 1 — Write the permutation test

1. Pool all 12 masses together
2. Randomly assign 5 of them to be "echolocators"
3. Compute the difference in means
4. Repeat 10,000 times
5. What fraction of random differences ≥ the observed difference?

**Plot** the null distribution as a histogram
and **mark** the observed difference with a red vertical line.

In [ ]:
# n_perm = 10000
# null_diffs = ...
#
# for i in range(n_perm):
#     ...
#
# p_value = ...
#
# # Plot the null distribution
# # Mark the observed difference
# # Print the p-value

### Exercise 2

1. Is the result significant at p < 0.05?
2. Try changing the echolocator masses to `[55, 60, 48, 52, 58]`.
   What happens to the p-value?
3. What if you only had 3 echolocators instead of 5? Does sample
   size matter?

In [ ]:
# Your code

---
## 4. Testing amino acids

Instead of comparing *means of body mass*, we now ask: at specific
positions in the prestin alignment, do echolocators share amino
acids more than expected by chance?

### 4.1 Load prestin alignment and species data

In [ ]:
prestin_seqs = read_fasta(os.path.join(SUB_DIR, 'SLC26A5.trim.fa'))
names = list(prestin_seqs.keys())
aln = np.array([list(seq) for seq in prestin_seqs.values()])
n_species, n_pos = aln.shape
print(f"Prestin alignment: {n_species} species × {n_pos} positions")

In [ ]:
species_df = pd.read_csv(os.path.join(DATA, 'species_classification.tsv'),
                         sep='\t', comment='#')
ecol = [c for c in species_df.columns if c.startswith('echo')][0]
species_df['taxid'] = species_df['taxid'].astype(str)
echo_lookup = dict(zip(species_df['taxid'], species_df[ecol]))

taxids = [name.split('.')[0] for name in names]
is_echo = np.array([echo_lookup.get(t, 'no') == 'yes' for t in taxids])
n_echo = int(is_echo.sum())
n_nonecho = n_species - n_echo
print(f"Echolocators:     {n_echo}")
print(f"Non-echolocators: {n_nonecho}")

### 4.2 Convergence score

At each position we ask: do echolocators agree on an amino acid
that non-echolocators **don't** share?

We define out 'metric':
1. Find the **most common amino acid among echolocators**
2. Compute its frequency in echolocators (echo_frac)
3. Compute its frequency in non-echolocators (nonecho_frac)
4. Convergence score = echo_frac - nonecho_frac

**High score** : echolocators converge on something others don't have.  
**Score close to 0** : both groups are similar (just conservation, not convergence).

In [ ]:
def convergence_score(echo_aas, nonecho_aas):
    """Differential agreement: echo vs non-echo for the top echo AA."""
    echo_filtered = [aa for aa in echo_aas if aa != '-']
    nonecho_filtered = [aa for aa in nonecho_aas if aa != '-']
    if not echo_filtered or not nonecho_filtered:
        return 0.0
    # Most common AA among echolocators
    top_aa, top_count = Counter(echo_filtered).most_common(1)[0]
    echo_frac = top_count / len(echo_filtered)
    # How common is that same AA among non-echolocators?
    nonecho_count = sum(1 for aa in nonecho_filtered if aa == top_aa)
    nonecho_frac = nonecho_count / len(nonecho_filtered)
    return echo_frac - nonecho_frac

In [ ]:
# Examples
print("Same AA everywhere:")
print(f"score = {convergence_score(['A']*5, ['A']*10):.2f}")
print()
print("Echolocators conserved A, non-echolocators mixed:")
print(f"score = {convergence_score(['A']*5, ['A','G','L','K','G','A','L','G','K','A']):.2f}")
print()
print("Both groups mixed:")
print(f"score = {convergence_score(['A','G','L','K','D'], ['A','G','L','K','D','A','G','L','K','D']):.2f}")

The first case (all identical) gives score close to 0 — that means
conservation, not convergence. The second case gives a high score —
echolocators specifically have A conserved while non-echolocators
are diverse.

### 4.3 Test one position

**Null hypothesis:** echolocator identity doesn't matter — a
random split of species into two groups of the same sizes would
give a similar score.

**->** shuffle which species are called "echolocator," recompute.

In [ ]:
pos = 150
# pos = 623

echo_aas = aln[is_echo, pos]
nonecho_aas = aln[~is_echo, pos]
obs_score = convergence_score(echo_aas, nonecho_aas)

print(f"Position {pos}")
print(f"Echo AAs:     {echo_aas}")
print(f"Non-echo AAs: {nonecho_aas}")
print(f"Convergence score: {obs_score:.3f}")

In [ ]:
n_perm = 10000
null_scores = np.zeros(n_perm)

for i in range(n_perm):
    perm = np.random.permutation(n_species)
    perm_echo = aln[perm[:n_echo], pos]
    perm_nonecho = aln[perm[n_echo:], pos]
    null_scores[i] = convergence_score(perm_echo, perm_nonecho)

pval = np.mean(null_scores >= obs_score)

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(null_scores, bins=40, color='steelblue', alpha=0.7,
        edgecolor='white')
ax.axvline(obs_score, color='red', linewidth=2, linestyle='--',
           label=f'Observed = {obs_score:.3f}')
ax.set_xlabel('Convergence score')
ax.set_ylabel('Count')
ax.set_title(f'Position {pos} — p = {pval:.4f}')
ax.legend()
plt.tight_layout()
plt.show()

Same logic as the balls and the body mass: compute a statistic,
shuffle labels, see how extreme the real value is.

### Exercise 3

Try a few other positions. Can you find one with a clearly
significant convergence score (p < 0.01) and one that isn't?
What amino acids do the echolocators share at the significant
position?

In [ ]:
# Your code here

---
## 5. Scanning the whole alignment

Now we test **every** position.

In [ ]:
n_perm = 1000
results = []

print(f"Scanning {n_pos} positions ({n_perm} permutations each)...")
for pos in range(n_pos):
    echo_aas = aln[is_echo, pos]
    nonecho_aas = aln[~is_echo, pos]
    obs = convergence_score(echo_aas, nonecho_aas)

    count = 0
    for _ in range(n_perm):
        perm = np.random.permutation(n_species)
        s = convergence_score(aln[perm[:n_echo], pos],
                              aln[perm[n_echo:], pos])
        if s >= obs:
            count += 1

    results.append({'position': pos, 'score': obs,
                    'pvalue': count / n_perm})
    

    if (pos + 1) % 100 == 0:
        print(f"  {pos + 1} / {n_pos}")

results_df = pd.DataFrame(results)
print("Finished")

In [ ]:
results_df[results_df['pvalue'] < 0.05]

In [ ]:
n_sig = (results_df['pvalue'] < 0.05).sum()
n_sig_strict = (results_df['pvalue'] < 0.01).sum()
print(f"Significant at p < 0.05:  {n_sig} / {n_pos}")
print(f"Significant at p < 0.01:  {n_sig_strict} / {n_pos}")

**Multiple testing.** We test hundreds of positions, some may
appear "significant" by chance. Testing 1000 positions at
p < 0.05, we expect ~50 false positives. Methods like Bonferroni
correction (divide threshold by number of tests) address this.

### 5.1 Manhattan plot

In [ ]:
def shannon_entropy(column):
    residues = [aa for aa in column if aa != '-']
    if not residues:
        return 0.0
    counts = Counter(residues)
    total = len(residues)
    return -sum((c/total) * math.log2(c/total) for c in counts.values())

entropies = [shannon_entropy(list(aln[:, p])) for p in range(n_pos)]

In [ ]:
pvals = results_df['pvalue'].replace(0, 1 / (n_perm + 1))
neg_log_p = -np.log10(pvals)

fig, axes = plt.subplots(2, 1, figsize=(14, 6), sharex=True)

axes[0].fill_between(range(n_pos), entropies, alpha=0.4, color='steelblue')
axes[0].plot(range(n_pos), entropies, linewidth=0.5, color='steelblue')
axes[0].set_ylabel('Shannon entropy')
axes[0].set_title('Conservation and convergence across prestin')

axes[1].scatter(range(n_pos), neg_log_p, s=3, c='steelblue', alpha=0.5)
axes[1].axhline(-np.log10(0.05), color='gray', linewidth=1, linestyle='--',
                label='p = 0.05')
# axes[1].axhline(-np.log10(0.01), color='red', linewidth=1, linestyle='--',
#                 label='p = 0.01')
axes[1].set_xlabel('Alignment position')
axes[1].set_ylabel('−log₁₀(p-value)')
axes[1].legend()

plt.tight_layout()
plt.show()

Positions above the line are **differentially conserved**
between echolocators and non-echolocators.

---
## 6. Summary

1. **Permutation tests** compute p-values by shuffling labels and
   comparing to a null distribution.
2. We applied this to colored balls, body mass, and amino acids.
3. At specific prestin positions, echolocators agree more than
   expected — evidence for **convergent evolution**.

**More rigorous analysis:**
- Phylogenetic correction (related species share AAs by descent)
- Ancestral reconstruction (true convergence vs shared ancestry)
- Correction for multiple testing

Parker et al. (2013) found genome-wide convergence across many
hearing genes.

---
## 7. Save results for the assignment

In [ ]:
results_path = os.path.join(DATA, 'prestin_convergence_results.csv')
results_df.to_csv(results_path, index=False)
print(f"Saved convergence results in {results_path}")

entropy_df = pd.DataFrame({'position': range(n_pos),
                           'entropy': entropies})
entropy_path = os.path.join(DATA, 'prestin_entropy.csv')
entropy_df.to_csv(entropy_path, index=False)
print(f"Saved entropy values in {entropy_path}")